# Build `truecost.db`
Creates the `prices`, `hospitals` and `codes` tables from the raw files in `data/`.

Run the cells top to bottom. The build step scans ~17 GB of CSV, so expect ~5-15 min.

In [ ]:
import sys, sqlite3
sys.path.insert(0, '../scripts')
import pandas as pd
import build_db
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)

## 1. Build the database (long-running)

In [ ]:
build_db.main()

## 2. Check the tables

In [ ]:
con = sqlite3.connect(build_db.DB)
for t in ['prices', 'hospitals', 'codes']:
    print(t, con.execute(f'select count(*) from {t}').fetchone()[0], 'rows')
pd.read_sql('select hospital, count(*) rows, count(distinct code) codes from prices group by hospital', con)

## 3. Example: code 77066 by insurer

In [ ]:
pd.read_sql("select hospital, payer, plan, negotiated_rate, cash_price, gross_charge from prices where code='77066' and payer is not null order by payer, negotiated_rate", con)

## 4. Coverage: which hospitals have which target codes

In [ ]:
pd.read_sql('select code, group_concat(distinct hospital) hospitals from prices group by code', con)

## 5. Other tables

In [ ]:
display(pd.read_sql('select * from hospitals', con)); pd.read_sql('select * from codes', con)